# Jacobians and Hessians

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 02.05 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/06_jacobians_and_hessians.ipynb)

---

## 🎯 Learning Objective

Generalize the derivative to vector-valued functions (Jacobians) and second derivatives (Hessians), and use Taylor expansion to locally approximate a loss surface.

---

## 📐 Theory

So far, derivatives have mapped a function to a number ($f'(a)$) or a vector ($\nabla f(a)$).
This notebook covers the two remaining generalizations: derivatives of *vector-valued*
functions, and *second* derivatives of scalar functions with many inputs.

### The Jacobian matrix

If $\mathbf{f}:\mathbb{R}^n \to \mathbb{R}^m$ outputs a vector
$\mathbf{f}(\mathbf{x}) = (f_1(\mathbf{x}), \dots, f_m(\mathbf{x}))$, its **Jacobian** collects
every output's gradient into one matrix:

$$J = \frac{\partial \mathbf{f}}{\partial \mathbf{x}} =
\begin{pmatrix} \frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{pmatrix} \in \mathbb{R}^{m\times n}$$

Row $i$ of $J$ is $\nabla f_i$ — the gradient of the $i$-th output. The Jacobian is the
*correct* multivariate generalization of "derivative": it's the matrix such that
$\mathbf{f}(\mathbf{x}+\mathbf{h}) \approx \mathbf{f}(\mathbf{x}) + J\mathbf{h}$, i.e. the best
*linear* approximation of $\mathbf{f}$ near $\mathbf{x}$, exactly mirroring $f(a+h)\approx
f(a)+f'(a)h$ from `02.02`. When $m=1$, $J$ is just $\nabla f$ written as a row vector.

### The chain rule for Jacobians

If $\mathbf{h}(\mathbf{x}) = \mathbf{g}(\mathbf{f}(\mathbf{x}))$, the multivariate chain rule
from `02.04` becomes a matrix product: $J_h(\mathbf{x}) = J_g(\mathbf{f}(\mathbf{x})) \cdot
J_f(\mathbf{x})$. Composing functions composes (multiplies) their Jacobians — this is precisely
how backpropagation through a stack of neural network layers works: each layer's local
Jacobian gets multiplied into the running product as gradients flow backward.

### The Hessian matrix

For a scalar function $f:\mathbb{R}^n\to\mathbb{R}$, the **Hessian** is the matrix of all
second partial derivatives — the "Jacobian of the gradient":

$$H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$$

By Clairaut's theorem (`02.03`), $H$ is symmetric ($H_{ij}=H_{ji}$) whenever $f$ is smooth
enough. $H$ describes **curvature**: its eigenvalues (recall `01.06`) tell you how $f$ bends
along each principal direction *at that point*, regardless of whether the point is special.

Turning eigenvalue signs into a **min / max / saddle classification of the point itself** —
the second-derivative test — requires $\mathbf{a}$ to be a **critical point**,
$\nabla f(\mathbf{a}) = \mathbf{0}$, first. At such a point: positive definite $H$ (every
eigenvalue $>0$) is *sufficient* for a strict local minimum; negative definite for a strict
local maximum; mixed signs mean $\mathbf{a}$ is a **saddle point** (a minimum along some
directions, a maximum along others, revisited in `04.01`). This is a sufficient, not a
necessary, condition: at an actual local minimum $H$ is only guaranteed to be positive
*semi*-definite (eigenvalues $\ge 0$), and a zero eigenvalue leaves the test inconclusive
(Exercise 2 works through exactly this case). Away from a critical point, a positive-definite
$H$ only means $f$ is locally convex near $\mathbf{a}$ — the point itself is on a slope, not a
minimum — a distinction the audit below checks for explicitly before labeling anything.

### Second-order Taylor expansion

Just as $f(a+h)\approx f(a)+f'(a)h$ is the first-order (linear) approximation, adding the next
term gives the second-order (quadratic) approximation, which for multivariate $f$ reads:

$$f(\mathbf{x}+\mathbf{h}) \approx f(\mathbf{x}) + \nabla f(\mathbf{x})^\top \mathbf{h} +
\frac{1}{2}\mathbf{h}^\top H(\mathbf{x}) \mathbf{h}$$

The gradient term captures the local slope; the Hessian term captures local curvature. This
quadratic model is exact for any function that's already a quadratic (like the loss surfaces
we've been plotting), and it's the foundation of Newton's method (`04.07`): rather than just
following the gradient downhill, jump straight to the minimum of this local quadratic
approximation.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Comparing a function's true surface against its first-order (linear) and second-order
(quadratic) Taylor approximations at a point shows visually how much curvature information
the Hessian adds: the quadratic model hugs the true surface much longer than the linear one.

In [ ]:
# f(x,y) = x^4 - 2x^2 + y^2 has a saddle-ish landscape; compare true surface vs 1st- and
# 2nd-order Taylor approximations at a point, as a 1D slice through y=y0 for readability.
f = lambda x, y: x**4 - 2*x**2 + y**2
grad_f = lambda x, y: np.array([4*x**3 - 4*x, 2*y])
hess_f = lambda x, y: np.array([[12*x**2 - 4, 0], [0, 2]])

x0, y0 = 1.3, 0.5
g, H = grad_f(x0, y0), hess_f(x0, y0)
x_line = np.linspace(x0 - 1.2, x0 + 1.2, 200)

true_vals = f(x_line, y0)
linear_approx = f(x0, y0) + g[0] * (x_line - x0)
quad_approx = f(x0, y0) + g[0] * (x_line - x0) + 0.5 * H[0, 0] * (x_line - x0) ** 2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(x_line, true_vals, color="#4C72B0", lw=2.5, label="true f(x, y=0.5)")
axes[0].plot(x_line, linear_approx, color="#DD8452", lw=1.5, ls="--", label="1st-order (linear)")
axes[0].plot(x_line, quad_approx, color="#55A868", lw=1.5, ls="--", label="2nd-order (quadratic)")
axes[0].plot(x0, f(x0, y0), 'ko', zorder=5)
axes[0].set_xlabel("x"); axes[0].set_ylabel("f(x, 0.5)")
axes[0].set_title("Quadratic (Hessian) approximation hugs the\ntrue curve far longer than linear")
axes[0].legend(fontsize=9)

# Eigenvalues of the Hessian at a few points. The min/max/saddle label is the
# second-derivative test, which classifies a point ONLY if it's a critical point (grad f = 0)
# -- (1.3, 0.5) is not one (it was just the Taylor-expansion point above), so we check
# stationarity numerically before applying the label, instead of assuming eigenvalue signs
# alone tell us the point's identity.
points = [(1.3, 0.5), (0.0, 0.0), (-1.0, 0.0)]
for x_p, y_p in points:
    g = grad_f(x_p, y_p)
    eigvals = np.linalg.eigvalsh(hess_f(x_p, y_p))
    is_critical = np.allclose(g, 0.0, atol=1e-8)
    if not is_critical:
        kind = "NOT a critical point (grad != 0) -- Hessian sign alone doesn't classify it"
    elif np.all(eigvals > 0):
        kind = "min-like (all +)"
    elif np.all(eigvals < 0):
        kind = "max-like (all -)"
    else:
        kind = "SADDLE (mixed signs)"
    print(f"Point ({x_p:+.1f}, {y_p:+.1f}): grad={np.round(g, 2)}, Hessian eigenvalues = {np.round(eigvals, 2)}  -> {kind}")

x_grid = np.linspace(-1.8, 1.8, 200); y_grid = np.linspace(-1.5, 1.5, 200)
Xg, Yg = np.meshgrid(x_grid, y_grid)
axes[1].contour(Xg, Yg, f(Xg, Yg), levels=20, cmap="viridis")
for x_p, y_p in points:
    axes[1].plot(x_p, y_p, 'o', markersize=8)
    axes[1].annotate(f"({x_p},{y_p})", (x_p, y_p), textcoords="offset points", xytext=(8, 8))
axes[1].set_title("f(x,y)=x^4-2x^2+y^2: two minima + one saddle at origin")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

We compute Jacobians and Hessians numerically from their definitions (nudge each coordinate,
per output/per pair of coordinates), verify the Jacobian chain rule on a composed function, and
confirm Hessian symmetry.

In [ ]:
def jacobian(f_vec, x, h=1e-5):
    """Numeric Jacobian of a vector-valued function f_vec: R^n -> R^m at point x."""
    x = np.array(x, dtype=float)
    f0 = np.array(f_vec(x))
    J = np.zeros((len(f0), len(x)))
    for j in range(len(x)):
        dx = np.zeros_like(x); dx[j] = h
        J[:, j] = (np.array(f_vec(x + dx)) - np.array(f_vec(x - dx))) / (2 * h)
    return J

def hessian(f_scalar, x, h=1e-4):
    """Numeric Hessian of a scalar function via finite differences of the gradient --
    literally 'the Jacobian of the gradient', matching the Theory section's framing."""
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            xpp, xpm, xmp, xmm = [np.array(x, dtype=float) for _ in range(4)]
            xpp[i] += h; xpp[j] += h
            xpm[i] += h; xpm[j] -= h
            xmp[i] -= h; xmp[j] += h
            xmm[i] -= h; xmm[j] -= h
            H[i, j] = (f_scalar(xpp) - f_scalar(xpm) - f_scalar(xmp) + f_scalar(xmm)) / (4 * h * h)
    return H

# Jacobian chain rule check: h(x) = g(f(x)) where f: R^2->R^2, g: R^2->R^2
f_vec = lambda x: np.array([x[0]**2, x[0] * x[1]])
g_vec = lambda y: np.array([np.sin(y[0]), y[1]**2])
h_vec = lambda x: g_vec(f_vec(x))

x0 = np.array([1.2, 0.7])
J_f = jacobian(f_vec, x0)
J_g = jacobian(g_vec, f_vec(x0))
J_h_composed = J_g @ J_f              # chain rule: J_h = J_g(f(x)) . J_f(x)
J_h_direct = jacobian(h_vec, x0)      # computed directly on the composed function

print("Jacobian of f at x0:\n", np.round(J_f, 4))
print("\nJ_h via chain rule (J_g @ J_f):\n", np.round(J_h_composed, 4))
print("\nJ_h computed directly on h=g(f(x)):\n", np.round(J_h_direct, 4))
print("\nMatch:", np.allclose(J_h_composed, J_h_direct, atol=1e-3))

# Hessian symmetry check on a non-trivial function
f_scalar = lambda x: x[0]**3 * x[1]**2 + np.exp(x[0] * x[1])
H = hessian(f_scalar, np.array([0.6, 0.9]))
print("\nHessian:\n", np.round(H, 4))
print("Symmetric (H == H.T):", np.allclose(H, H.T, atol=1e-2))
print("Eigenvalues:", np.round(np.linalg.eigvalsh(H), 3))

## 🤖 Why This Matters for AI

Every layer in a neural network is a vector-valued function, and PyTorch's backward pass is
literally the Jacobian chain rule from this notebook: each layer contributes its local
Jacobian, and reverse-mode autodiff multiplies them together (technically, multiplies
Jacobian-vector products, to avoid ever forming a full Jacobian matrix — full Jacobians for a
layer with millions of units would be far too large to store). The Hessian shows up wherever
optimization needs curvature: **second-order methods** (`04.07`) use it directly, and the
*sharpness* of minima found by SGD (flat vs. sharp, tied to Hessian eigenvalues) is an active
area of generalization research (`04.09`). Below, we use PyTorch's `functorch`-style
`torch.autograd.functional` utilities to compute a real Jacobian and Hessian for a small
network, and confirm they match our from-scratch numeric versions.

In [ ]:
import torch
from torch.autograd.functional import jacobian as torch_jacobian
from torch.autograd.functional import hessian as torch_hessian

# 1) Confirm torch.autograd's Jacobian matches our from-scratch numeric one, on the SAME f
def f_vec_torch(x):
    return torch.stack([x[0] ** 2, x[0] * x[1]])

x0_t = torch.tensor([1.2, 0.7])
J_torch = torch_jacobian(f_vec_torch, x0_t).numpy()
print("Jacobian (torch.autograd):\n", np.round(J_torch, 4))
print("Jacobian (our from-scratch):\n", np.round(J_f, 4))
print("Match:", np.allclose(J_torch, J_f, atol=1e-3))

# 2) Confirm torch.autograd's Hessian matches ours. The min/max/saddle test only classifies
#    a point once we've confirmed it's a critical point (grad = 0) -- that's the
#    second-derivative test's actual hypothesis, so we check it numerically first rather than
#    reading a point's identity off eigenvalue signs alone.
def f_scalar_torch(x):
    return x[0] ** 3 * x[1] ** 2 + torch.exp(x[0] * x[1])

x1_t = torch.tensor([0.6, 0.9])
grad_torch = torch_jacobian(f_scalar_torch, x1_t).numpy()  # for scalar output, Jacobian IS the gradient
H_torch = torch_hessian(f_scalar_torch, x1_t).numpy()
print("\nGradient (torch.autograd):", np.round(grad_torch, 4))
print("Hessian (torch.autograd):\n", np.round(H_torch, 4))
print("Hessian (our from-scratch):\n", np.round(H, 4))
print("Match:", np.allclose(H_torch, H, atol=1e-2))

eigvals = np.linalg.eigvalsh(H_torch)
is_critical = np.allclose(grad_torch, 0.0, atol=1e-6)
print(f"\nEigenvalues: {np.round(eigvals, 3)}")
print(f"Is (0.6, 0.9) a critical point (grad ~ 0)? {is_critical}")
if not is_critical:
    print("Not a critical point, so the min/max/saddle test doesn't apply here -- mixed-sign\n"
          "eigenvalues only describe local curvature (concave along one direction, convex along\n"
          "another) AT this point, not the identity of the point itself.")
elif np.all(eigvals > 0):
    print("All positive -> this point sits in a convex (bowl-shaped) region: a local minimum.")
elif np.all(eigvals < 0):
    print("All negative -> local maximum.")
else:
    print("Mixed signs -> SADDLE POINT: Newton's method without care would step toward it, not away.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $\mathbf{f}(x,y) = (x^2y,\ xy^2)$, compute the $2\times2$ Jacobian by hand at
   $(x,y)=(1,2)$. Verify with `jacobian`, and confirm the shape is $2\times2$ (outputs $\times$
   inputs), matching $\mathbf{f}:\mathbb{R}^2\to\mathbb{R}^2$.

<details>
<summary>💡 Solution</summary>

$f_1=x^2y$, $f_2=xy^2$. Row 1 is $\nabla f_1 = (2xy,\ x^2)$; row 2 is
$\nabla f_2 = (y^2,\ 2xy)$. So
$$J = \begin{pmatrix} 2xy & x^2 \\ y^2 & 2xy \end{pmatrix}, \qquad
J(1,2) = \begin{pmatrix} 4 & 1 \\ 4 & 4 \end{pmatrix}.$$
`jacobian(f_vec, (1,2))` reproduces this numerically (`[[4. 1.] [4. 4.]]`) to well within
finite-difference tolerance. The shape is $2\times2$ because $\mathbf{f}$ has 2 outputs and 2
inputs — row $i$, column $j$ is always $\partial f_i/\partial x_j$, so the Jacobian's shape is
always (number of outputs) $\times$ (number of inputs), regardless of whether the two happen to
coincide.

</details>

### 💭 UNDERSTAND
2. For $f(x,y) = x^2+y^4$, the Hessian at $(0,0)$ has eigenvalues $\{2, 0\}$ — inconclusive for
   the second-derivative test, since a zero eigenvalue means the quadratic (Hessian-only) model
   can't see what the $y^4$ term does. Compare it against $g(x,y)=x^2-y^4$, whose Hessian at
   $(0,0)$ has the *identical* eigenvalue pattern $\{2, 0\}$ but a completely different answer:
   is $(0,0)$ a minimum of $g$? Resolve both cases by reasoning about each function directly
   (not through the Hessian), and explain what this pair proves about the second-derivative
   test's limits.

<details>
<summary>💡 Solution</summary>

For $f(x,y)=x^2+y^4$: since $x^2\ge0$ and $y^4\ge0$ always, $f(x,y)\ge0=f(0,0)$ everywhere, so
$(0,0)$ **is** a genuine (global) minimum — the Hessian's zero eigenvalue just means the
*quadratic* model alone can't certify it (the quartic term, invisible to a Hessian, is what
actually enforces it). For $g(x,y)=x^2-y^4$: along the line $x=0$, $g(0,y)=-y^4<0$ for any
$y\ne0$, while $g(0,0)=0$ — so $(0,0)$ is **not** a minimum of $g$ (it's a saddle: increasing
along the $x$-direction, decreasing along the $y$-direction, just more slowly than a quadratic
would). Both functions have exactly the same Hessian eigenvalues $\{2,0\}$ at the origin, yet
one point is a genuine minimum and the other is a saddle — direct, concrete proof that a zero
eigenvalue really does make the test inconclusive, not just technically-but-rarely-in-practice
so: the *sign* of the quartic term, information the Hessian (a purely quadratic/second-order
object) cannot see at all, is what actually decides the answer here.

</details>

### ✏️ DERIVE
3. Derive the Jacobian chain-rule identity $J_h = J_g(\mathbf{f}(\mathbf{x}))\cdot
   J_f(\mathbf{x})$ for the specific composition $\mathbf{h}(x,y)=\mathbf{g}(\mathbf{f}(x,y))$
   where $\mathbf{f}(x,y)=(x+y,\ xy)$ and $\mathbf{g}(u,v)=(u^2,\ \sin v)$: compute $J_f$, $J_g$
   (as a function of $u,v$), substitute $\mathbf{f}(x,y)$ into $J_g$, multiply the two matrices,
   and confirm the result matches differentiating $\mathbf{h}$'s two components directly.

<details>
<summary>💡 Solution outline</summary>

$J_f = \begin{pmatrix}1&1\\y&x\end{pmatrix}$ (rows: $\nabla(x+y)=(1,1)$, $\nabla(xy)=(y,x)$).
$J_g = \begin{pmatrix}2u&0\\0&\cos v\end{pmatrix}$ (diagonal, since each of $g_1,g_2$ depends on
only one of $u,v$). Substituting $u=x+y$, $v=xy$:
$J_g(\mathbf{f}(x,y)) = \begin{pmatrix}2(x+y)&0\\0&\cos(xy)\end{pmatrix}$. Multiplying:
$$J_h = J_g(\mathbf{f})\cdot J_f =
\begin{pmatrix}2(x+y)&0\\0&\cos(xy)\end{pmatrix}\begin{pmatrix}1&1\\y&x\end{pmatrix} =
\begin{pmatrix}2(x+y)&2(x+y)\\y\cos(xy)&x\cos(xy)\end{pmatrix}.$$
Directly: $h_1(x,y)=(x+y)^2$, so $\nabla h_1=(2(x+y),\,2(x+y))$; $h_2(x,y)=\sin(xy)$, so
$\nabla h_2 = (y\cos(xy),\,x\cos(xy))$ — identical to the chain-rule product, row by row. At
$(x,y)=(1.5,-0.5)$ both routes give $\begin{pmatrix}2&2\\-0.3658&1.0975\end{pmatrix}$ to 4
decimal places, confirming composing functions really does correspond to *multiplying* their
Jacobians, matrix-fashion, exactly as backpropagation relies on when chaining layers (`02.04`,
`02.10`).

</details>

### 🐍 IMPLEMENT
4. For a "residual" function $\mathbf{r}:\mathbb{R}^2\to\mathbb{R}^2$ and loss
   $f(\mathbf{x})=\|\mathbf{r}(\mathbf{x})\|^2$, implement the **Gauss-Newton approximation**
   to the Hessian, $H_{GN} = 2J_r^\top J_r$ (using only `jacobian`, no second derivatives of
   $\mathbf{r}$ at all), for $\mathbf{r}(x,y)=(x^2-y,\ xy-1)$. Compare $H_{GN}$ against the true
   `hessian(f, ...)` at a generic point, and again at a point where $\mathbf{r}\approx\mathbf{0}$.

<details>
<summary>✅ How to know you're right</summary>

$H_{GN}$ should always come out symmetric and positive-semi-definite (both guaranteed by its
$J^\top J$ form — check `np.allclose(H_gn, H_gn.T)` and that its eigenvalues are $\ge0$,
regardless of what $\mathbf{r}$ is). At a generic point (e.g. $(1.3,0.8)$, where
$\mathbf{r}\approx(0.89,0.04)$, not small), $H_{GN}$ and the true Hessian will *disagree*
noticeably (by roughly $3$–$4$ in the largest entry in one check) — this is expected, not a
bug, since Gauss-Newton drops a genuine second-order term involving $\mathbf{r}$ itself. At a
point where $\mathbf{r}(\mathbf{x})\approx\mathbf{0}$ (e.g. $(1,1)$, where
$\mathbf{r}(1,1)=(0,0)$ exactly), $H_{GN}$ should match the true Hessian almost exactly (to
within finite-difference tolerance, around $10^{-6}$) — the dropped term is proportional to
$\mathbf{r}$ itself, so it vanishes exactly where the residual does. If your two Hessians agree
equally well (or equally badly) at both points, the bug is likely a wrong Jacobian shape in the
$J^\top J$ product rather than a fundamental issue with the approximation.

</details>

### 🤖 APPLY
5. For a small linear-plus-nonlinearity layer $\text{layer}(\mathbf{x})=\tanh(W\mathbf{x})$
   ($W\in\mathbb{R}^{3\times5}$, fixed), compute the Jacobian at a random input $\mathbf{x}_0$
   and its singular values. The **largest** singular value bounds how much the output can move
   (to first order) per unit-norm input perturbation. Verify this bound by comparing the output
   shift from perturbing $\mathbf{x}_0$ by $\epsilon$ along the top right-singular-vector
   direction against perturbing by the same $\epsilon$ in 2,000 random unit directions.

<details>
<summary>✅ What you should observe</summary>

The output shift along the top singular vector direction should closely match
$\epsilon\times(\text{largest singular value})$ (in one run: predicted $\approx0.01501$,
observed $\approx0.01496$ — matching to within the expected first-order/linearization error at
$\epsilon=0.01$), and it should be **at least as large** as the output shift from any of the
2,000 random directions tried (observed random-direction shifts topped out around $0.01484$,
below the singular-vector direction's $0.01496$, with the *average* random shift roughly half
that). This is the Jacobian-based analogue of `02.05`'s single-output steepest-direction result,
generalized to vector-valued outputs: the top singular vector of $J$ is the *worst-case* input
direction for a Lipschitz/robustness bound on this layer, and its singular value is exactly the
layer's local Lipschitz constant — the same quantity that shows up in adversarial robustness
analysis and in diagnosing exploding activations/gradients layer by layer.

</details>

### 🚀 CHALLENGE
6. Build a 2-layer `torch` network (`Linear(3,4) -> tanh -> Linear(4,1)`) and use
   `torch.autograd.functional.hessian` to compute the Hessian of its scalar output with respect
   to its 3 *inputs* (not its weights) at a random input point. Report the eigenvalues and
   classify the point as min-like, max-like, or a saddle **only after** checking whether it's
   actually a critical point. Then explain why, for a real network with millions of
   *parameters* rather than 3 inputs, forming this full Hessian matrix (which would be
   millions-by-millions) is intractable in practice, and what practitioners use instead
   (connect to the Gauss-Newton approximation from IMPLEMENT, and to `04.07`).

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer checks $\nabla(\text{output})$ at the random input point first, finds it is (in
general, for an arbitrary random point) *not* close to $\mathbf{0}$, and correctly concludes
the min/max/saddle classification doesn't strictly apply — the Hessian's eigenvalue signs at a
non-critical point describe local curvature only (this notebook's own Implementation and AI
sections make exactly this distinction, and a strong answer explicitly re-applies it here
rather than mechanically reporting "saddle" from mixed eigenvalue signs alone). For the
scaling question: a Hessian for $p$ parameters has $p^2$ entries — for a network with even a
modest 10 million parameters, that's $10^{14}$ entries, far beyond any realistic memory budget,
and computing it via repeated Hessian-vector products would still need $O(p)$ backward passes.
A strong answer names at least one real workaround: Gauss-Newton/Fisher-information
approximations (IMPLEMENT's $H_{GN}=2J^\top J$, generalized to the natural gradient method
cited in Further Reading) that never form the full Hessian, and/or Hessian-vector products
(computed via two backward passes, without ever materializing $H$ itself) used by practical
second-order and curvature-aware optimizers — both sidestep the $O(p^2)$ storage wall by only
ever needing $H$ applied to a vector, not $H$ itself.

</details>

---

## 📚 Further Reading
- Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 5.6–5.9 (Jacobian, Hessian, Taylor Series)
- Martens, ["New Insights and Perspectives on the Natural Gradient Method"](https://arxiv.org/abs/1412.1193) (curvature-aware optimization)

---

*Next notebook: [Integration Fundamentals](07_integration_fundamentals.ipynb)*